# MONAI Lung CT Registration - Batch Inference

Run distributed batch inference using the trained MONAI model on Snowflake GPU compute pools.

## Prerequisites
- Complete `03_train_and_register.ipynb` first
- Model logged to registry: `MONAI_LUNG_CT_REGISTRATION/v1`

In [1]:
!pip install -q monai nibabel torch snowflake-ml-python

In [2]:
import os
import tempfile
import gzip
import shutil

import numpy as np
import pandas as pd
import nibabel as nib
import torch
import torch.nn.functional as F

from snowflake.snowpark import Session
from snowflake.ml.registry import Registry

print(f"PyTorch: {torch.__version__}")

/opt/anaconda3/envs/snowml-local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.5.1


## Configuration

In [24]:
# Connection name from ~/.snowflake/connections.toml
# See: https://docs.snowflake.com/en/developer-guide/snowpark/python/creating-session
CONNECTION_NAME = "monai-quickstart"  # Change to your connection name

# Snowflake objects created by 01_setup.sql
ROLE = "MONAI_USER"
WAREHOUSE = "MONAI_WH"
DATABASE = "MONAI_QUICKSTART_DB"
SCHEMA = "ML"

# Same preprocessing as training
SPATIAL_SIZE = (96, 96, 96)
CT_MIN_HU = -1000
CT_MAX_HU = 500

In [25]:
# Create session using connection from connections.toml
session = Session.builder.config("connection_name", CONNECTION_NAME).create()

# Switch to MONAI resources
session.use_role(ROLE)
session.use_warehouse(WAREHOUSE)
session.use_database(DATABASE)
session.use_schema(SCHEMA)

print(f"Connected: {session.get_current_database()}.{session.get_current_schema()}")

Connected: "MONAI_QUICKSTART_DB"."ML"


## Load Model from Registry

In [5]:
registry = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)

MODEL_NAME = "MONAI_LUNG_CT_REGISTRATION"
VERSION = "v1"

mv = registry.get_model(MODEL_NAME).version(VERSION)
print(f"Loaded: {mv.fully_qualified_model_name}")

# Verify expected input shape
funcs = mv.show_functions()
sig = funcs[0]['signature']
print(f"Input shapes: {[(f.name, f._shape) for f in sig.inputs]}")

Loaded: MONAI_QUICKSTART_DB.ML.MONAI_LUNG_CT_REGISTRATION
Input shapes: [('input_feature_0', (96, 96, 96)), ('input_feature_1', (96, 96, 96))]


## Prepare Batch Input Data

The model expects 3D volumetric data. For batch inference, we:
1. Load images from stage
2. Preprocess (normalize, resize) to match training
3. Create input DataFrame with nested 3D arrays

In [6]:
LOCAL_DATA_DIR = tempfile.mkdtemp()
STAGE = "@MEDICAL_IMAGES_STG"

# Download test images (scans folder contains both exp and insp)
scans_folder = os.path.join(LOCAL_DATA_DIR, "scans")
os.makedirs(scans_folder, exist_ok=True)
session.file.get(f"{STAGE}/scans/", scans_folder)

# Handle double-gzip if present
for f in os.listdir(scans_folder):
    if f.endswith('.gz.gz'):
        gz_path = os.path.join(scans_folder, f)
        out_path = gz_path[:-3]
        with gzip.open(gz_path, 'rb') as f_in:
            with open(out_path, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        os.remove(gz_path)

print(f"Downloaded to: {LOCAL_DATA_DIR}")

Downloaded to: /var/folders/wh/1t1qbkgx1csgf4q8fxkt5gnr0000gn/T/tmpsx8_s65l


In [8]:
def preprocess_ct(nifti_path, spatial_size):
    """Load and preprocess CT scan matching training pipeline."""
    img = nib.load(nifti_path)
    data = img.get_fdata().astype(np.float32)
    
    # HU normalization (same as training: ScaleIntensityRanged)
    data = np.clip(data, CT_MIN_HU, CT_MAX_HU)
    data = (data - CT_MIN_HU) / (CT_MAX_HU - CT_MIN_HU)
    
    # Resize (same as training: Resized with trilinear mode)
    tensor = torch.from_numpy(data).unsqueeze(0).unsqueeze(0)  # [1, 1, D, H, W]
    tensor = F.interpolate(tensor, size=spatial_size, mode='trilinear', align_corners=True)
    
    return tensor.squeeze(0)  # [1, D, H, W]


# Build batch input
batch_data = []
num_cases = 10  # Adjust as needed

for i in range(1, num_cases + 1):
    case_id = f"case_{i:03d}"
    
    fixed_path = f"{LOCAL_DATA_DIR}/scans/{case_id}_exp.nii.gz"
    moving_path = f"{LOCAL_DATA_DIR}/scans/{case_id}_insp.nii.gz"
    
    fixed = preprocess_ct(fixed_path, SPATIAL_SIZE)
    moving = preprocess_ct(moving_path, SPATIAL_SIZE)
    
    # Model expects [batch, 2, D, H, W] - split into input features
    # input_feature_0: moving image [D, H, W]
    # input_feature_1: fixed image [D, H, W]
    batch_data.append({
        "input_feature_0": moving.squeeze(0).numpy().tolist(),
        "input_feature_1": fixed.squeeze(0).numpy().tolist(),
        "CASE_ID": case_id
    })
    print(f"Processed: {case_id}")

print(f"Batch ready: {len(batch_data)} cases")

Processed: case_001
Processed: case_002
Processed: case_003
Processed: case_004
Processed: case_005
Processed: case_006
Processed: case_007
Processed: case_008
Processed: case_009
Processed: case_010
Batch ready: 10 cases


## Upload Batch Input

Write the batch input to a stage as Parquet for efficient processing.

In [9]:
BATCH_INPUT_STAGE = f"@{DATABASE}.{SCHEMA}.BATCH_INPUT_STG"
BATCH_OUTPUT_STAGE = f"@{DATABASE}.{SCHEMA}.BATCH_OUTPUT_STG"

# Create stages
session.sql(f"CREATE STAGE IF NOT EXISTS BATCH_INPUT_STG").collect()
session.sql(f"CREATE STAGE IF NOT EXISTS BATCH_OUTPUT_STG").collect()

# Write to local parquet, upload to stage
input_df = pd.DataFrame({
    "INPUT_FEATURE_0": [d["input_feature_0"] for d in batch_data],
    "INPUT_FEATURE_1": [d["input_feature_1"] for d in batch_data],
    "CASE_ID": [d["CASE_ID"] for d in batch_data]
})

temp_parquet = tempfile.NamedTemporaryFile(suffix=".parquet", delete=False).name
input_df.to_parquet(temp_parquet, engine='pyarrow')
session.file.put(temp_parquet, BATCH_INPUT_STAGE, auto_compress=False, overwrite=True)

# Read back as Snowpark DataFrame
parquet_name = os.path.basename(temp_parquet)
batch_input_df = session.read.parquet(f"{BATCH_INPUT_STAGE}/{parquet_name}")

print(f"Input DataFrame: {batch_input_df.count()} rows")
os.unlink(temp_parquet)

Input DataFrame: 10 rows


## Run Batch Inference

Execute distributed inference on GPU compute pool.

In [ ]:
from snowflake.ml.model.batch import OutputSpec

COMPUTE_POOL = "MONAI_GPU_POOL"

print(f"Starting batch inference on {COMPUTE_POOL}...")
print(f"  Model: {MODEL_NAME}/{VERSION}")
print(f"  Cases: {len(batch_data)}")

job = mv.run_batch(
    compute_pool=COMPUTE_POOL,
    X=batch_input_df,
    output_spec=OutputSpec(stage_location=BATCH_OUTPUT_STAGE),
)

# Wait for job to complete
job.wait()
print("Batch inference complete!")

Starting batch inference on MONAI_GPU_POOL...
  Model: MONAI_LUNG_CT_REGISTRATION/v1
  Cases: 10
Batch inference complete!


## Read Results

In [22]:
# Create file format and list output files
session.sql("CREATE FILE FORMAT IF NOT EXISTS PARQUET_FF TYPE = PARQUET").collect()

output_files = session.sql(f"LIST {BATCH_OUTPUT_STAGE}").to_pandas()
parquet_files = output_files[output_files.iloc[:, 0].str.endswith('.parquet')]

# Get case IDs from results
cases_df = session.sql(f"""
    SELECT $1:CASE_ID::STRING AS CASE_ID, METADATA$FILENAME AS OUTPUT_FILE
    FROM {BATCH_OUTPUT_STAGE}
    (FILE_FORMAT => PARQUET_FF, PATTERN => '.*\\.parquet')
""").to_pandas()

# Display results
print("Batch Inference Results")
print("="*50)

for _, row in cases_df.iterrows():
    case_id = row['CASE_ID']
    output_file = row['OUTPUT_FILE'].split('/')[-1]
    print(f"{case_id} -> {output_file}")

print("="*50)
print(f"Total: {len(cases_df)} cases processed")

Batch Inference Results
case_006 -> 3_8e91864dace8496d9f32f5c745271cca_000005_000000-0.parquet
case_001 -> 3_8e91864dace8496d9f32f5c745271cca_000000_000000-0.parquet
case_005 -> 3_8e91864dace8496d9f32f5c745271cca_000004_000000-0.parquet
case_007 -> 3_8e91864dace8496d9f32f5c745271cca_000006_000000-0.parquet
case_008 -> 3_8e91864dace8496d9f32f5c745271cca_000007_000000-0.parquet
case_002 -> 3_8e91864dace8496d9f32f5c745271cca_000001_000000-0.parquet
case_009 -> 3_8e91864dace8496d9f32f5c745271cca_000008_000000-0.parquet
case_004 -> 3_8e91864dace8496d9f32f5c745271cca_000003_000000-0.parquet
case_010 -> 3_8e91864dace8496d9f32f5c745271cca_000009_000000-0.parquet
case_003 -> 3_8e91864dace8496d9f32f5c745271cca_000002_000000-0.parquet
Total: 10 cases processed


In [27]:
# Examine one sample result to verify inference output
import json

sample = session.sql(f"""
    SELECT $1 AS result
    FROM {BATCH_OUTPUT_STAGE}
    (FILE_FORMAT => PARQUET_FF, PATTERN => '.*\\.parquet')
    LIMIT 1
""").to_pandas()

result = json.loads(sample['RESULT'].iloc[0])

print("Sample Inference Output")
print("="*50)
print(f"Case ID: {result.get('CASE_ID', 'N/A')}")
print()

# Get output features (DDF channels)
output_keys = sorted([k for k in result.keys() if k.startswith('output_feature')])
print(f"Output features: {len(output_keys)}")

for key in output_keys:
    ddf_channel = np.array(result[key])
    print(f"  {key}: shape {ddf_channel.shape}, "
          f"mean={ddf_channel.mean():.4f}, std={ddf_channel.std():.4f}")

print()
print("Inference completed successfully.")

Sample Inference Output
Case ID: case_008

Output features: 3
  output_feature_0: shape (96, 96, 96), mean=0.0643, std=0.6380
  output_feature_1: shape (96, 96, 96), mean=0.1420, std=0.4365
  output_feature_2: shape (96, 96, 96), mean=0.2164, std=0.2084

Inference completed successfully.


In [23]:
session.close()
print("Done!")

Done!
